# PEAK M-ATH — Cluster Process Parent-Child Execution Chains

Cluster process parent-child chains across the endpoint fleet; identify rare or never-seen process trees that deviate from the norm, following the **PEAK** framework: **Prepare → Execute → Act → Knowledge**.

**Ref:** M08 — Cluster process parent-child execution chains  
**M-ATH approach:** Fleet-wide frequency counting of parent-child process basename pairs to surface rare execution chains.

## How to use
1. Put EDR CSV files into `input/` (with parent/child process columns, e.g. src.process.parent.image.path, src.process.image.path)
2. Run all cells
3. Review rare pairs in `output/`

In [ ]:
pass  # Placeholder (removed environment-specific output)

## PREPARE — Plan your Approach

- **Select Topic:** Rare process parent-child execution chains — adversaries spawn unusual child processes from trusted parents to blend in (MITRE ATT&CK [T1059](https://attack.mitre.org/techniques/T1059/) Command and Scripting Interpreter, [T1036](https://attack.mitre.org/techniques/T1036/) Masquerading).
- **Research Topic:** Process lineage analysis, fleet-wide baseline of normal parent-child relationships, rarity-based anomaly detection.
- **Identify Datasets:** EDR process telemetry with parent and child process paths/names and endpoint identifiers.
- **Select Algorithms:** Frequency counting of parent-child basename pairs across the fleet; rarity thresholding (percentile-based and minimum occurrence filters).

In [ ]:
# Scenario mode: run from scenarios/cluster_process_parent_child_execution_chains/
import os
import sys
from pathlib import Path
SCENARIO_DIR = Path('.').resolve()
REPO_ROOT = SCENARIO_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
INPUT_DIR = SCENARIO_DIR / 'input'
OUTPUT_DIR = SCENARIO_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIO_MODE = True

In [ ]:
import glob
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 120)

if not globals().get('SCENARIO_MODE', False):
    INPUT_DIR = Path('input')
    OUTPUT_DIR = Path('output')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _rel(p):
    try:
        if globals().get('SCENARIO_MODE', False) and 'REPO_ROOT' in globals() and hasattr(p, 'is_relative_to') and p.is_relative_to(REPO_ROOT):
            return p.relative_to(REPO_ROOT)
        return p
    except (ValueError, AttributeError):
        return p

print(f'Input folder: {_rel(INPUT_DIR)}')
print(f'Output folder: {_rel(OUTPUT_DIR)}')

## EXECUTE — Experimentation Time

- **Gather Data:** Load EDR CSVs from `input/`, detect parent/child process and endpoint columns.
- **Pre-Process Data:** Extract process basenames, normalize paths, drop incomplete records.
- **Apply:** Count occurrences and unique endpoints per parent-child pair; flag rare pairs below rarity thresholds.
- **Analyze:** Review rare pairs sorted by occurrence count and endpoint spread.
- **Escalate Critical Findings:** Never-before-seen parent-child chains on multiple endpoints warrant immediate investigation for lateral movement or persistence.

## Load EDR process data

In [ ]:
csv_paths = sorted(glob.glob(str(INPUT_DIR / '**' / '*.csv'), recursive=True))
print(f'Found {len(csv_paths)} CSV file(s).')

if not csv_paths:
    raise FileNotFoundError(f'No CSV files in {_rel(INPUT_DIR)}. Add EDR logs with parent/child process columns.')

dfs = []
for p in csv_paths:
    df = pd.read_csv(p)
    try:
        src_rel = str(Path(p).relative_to(REPO_ROOT)) if 'REPO_ROOT' in globals() and Path(p).is_relative_to(REPO_ROOT) else p
    except (ValueError, AttributeError):
        src_rel = p
    df['_source_file'] = src_rel
    dfs.append(df)
raw = pd.concat(dfs, ignore_index=True)

parent_col = next((c for c in raw.columns if 'parent' in c.lower() and ('image' in c.lower() or 'path' in c.lower() or 'name' in c.lower())), None)
child_col = next((c for c in raw.columns if 'parent' not in c.lower() and ('image.path' in c or 'process.path' in c or 'process.name' in c)), None)
if not parent_col:
    parent_col = next((c for c in raw.columns if 'parent' in c.lower()), None)
if not child_col:
    child_col = next((c for c in raw.columns if 'process' in c.lower() and 'path' in c.lower()), None)
if not child_col:
    child_col = next((c for c in raw.columns if 'process' in c.lower() and 'name' in c.lower()), None)

if not parent_col or not child_col:
    raise ValueError(f'Need parent and child process columns. Found: {list(raw.columns)}')

endpoint_col = next((c for c in raw.columns if 'agent.uuid' in c or 'endpoint.name' in c or 'host' in c.lower()), 'agent.uuid')
if endpoint_col not in raw.columns:
    raw['_endpoint'] = raw.index.astype(str)
    endpoint_col = '_endpoint'

raw = raw.rename(columns={parent_col: 'parent', child_col: 'child', endpoint_col: 'endpoint'})
raw = raw[['parent', 'child', 'endpoint']].dropna(subset=['parent', 'child'])
raw['parent'] = raw['parent'].astype(str).str.strip()
raw['child'] = raw['child'].astype(str).str.strip()
raw = raw[(raw['parent'] != '') & (raw['child'] != '') & (raw['parent'].str.lower() != 'null')]

def basename(path: str) -> str:
    s = str(path).strip()
    if '\\' in s:
        return s.split('\\')[-1].lower()
    if '/' in s:
        return s.split('/')[-1].lower()
    return s.lower()

raw['parent_name'] = raw['parent'].apply(basename)
raw['child_name'] = raw['child'].apply(basename)
print(f'Loaded {len(raw):,} process creation records.')

## Build frequency counts and flag rare pairs

In [ ]:
pair_counts = raw.groupby(['parent_name', 'child_name']).agg(
    total_occurrences=('endpoint', 'count'),
    unique_endpoints=('endpoint', 'nunique')
).reset_index()

total_endpoints = raw['endpoint'].nunique()
pair_counts['pct_endpoints'] = (pair_counts['unique_endpoints'] / total_endpoints * 100).round(2)

RARE_PCT = 1.0  # Flag pairs seen on <1% of endpoints
MIN_OCCURRENCES = 2  # Require at least 2 occurrences to consider

rare = pair_counts[
    (pair_counts['pct_endpoints'] < RARE_PCT) &
    (pair_counts['unique_endpoints'] >= 1)
].copy()
rare = rare.sort_values('pct_endpoints').reset_index(drop=True)

print(f'Total endpoints: {total_endpoints:,}')
print(f'Unique (parent, child) pairs: {len(pair_counts):,}')
print(f'Rare pairs (seen on <{RARE_PCT}% of endpoints): {len(rare):,}')
if len(rare) > 0:
    display(rare.head(30))

## ACT — Wrapping Up the Investigation

- **Document Findings:** Rare parent-child pairs with occurrence counts and endpoint spread exported for triage.
- **Preserve Hunt:** Results archived to `output/process_lineage_rare_pairs.csv`.
- **Create Detections / Playbooks:** Confirmed anomalous execution chains can feed endpoint detection rules or SOAR playbooks for automated containment.

In [ ]:
out_path = OUTPUT_DIR / 'process_lineage_rare_pairs.csv'
rare.to_csv(out_path, index=False)
print(f'Saved to {_rel(out_path)}')

## KNOWLEDGE — Continuous Improvement

- **Re-Add Topics to Backlog:** New execution chain patterns or living-off-the-land techniques discovered during analysis become future hunt hypotheses.
- **Communicate Findings:** Share rare execution chains with SOC, endpoint security, and application owners for root-cause analysis.
- **Feed Improvements Back:** Add confirmed benign rare pairs to allowlists; tune rarity thresholds as the fleet evolves; consider vectorization or clustering for deeper lineage analysis.
- **Measure Effectiveness:** Track rare-pair counts and confirmed true positives across runs to assess detection value.